## Predicting Crop Yield

> The clean table generated in the `marts` stage has historical data that cannot tell a programme officer what yield to expect from a specific farm in a given season. Additionally, the data in the clean table cannot tell us which factors drive crop yield. This stage trains a model to predict crop yield, tests it on unseen data, and identifies the factors that have the largest impact on crop yield.

> The model reads from `marts.yield_dataset` and its predictions are written back to the database. **Steps followed:**

1.  Load the usable rows and select relevant features.
1.   **Train-test split:** Split dataset by time: train on 2019 to 2022 data and test on 2023 to 2024 data.
1.   Compare four models using cross-validation on the training data only: a simple baseline, linear regression, random forest and gradient boosting.
2.   Evaluate the best performing model once on the test data.
2.   Explain the results: which features matter, and what shape the rainfall and fertilizer effects take.
1.   Save the model and write the predictions back to the DuckDB database.













### Establish connections.

In [3]:
from pathlib import Path
from google.colab import drive
import shutil
import duckdb

# Mount Google Drive
drive.mount("/content/drive")

# Define paths
drive_db = Path("/content/drive/MyDrive/Agriculture_ELT/data/agri.duckdb")
db_path = Path("/content/agri.duckdb")

# Load the latest saved database from Drive
assert drive_db.exists(), "Database not found in Google Drive"

if "conn" in globals():
    conn.close()

if db_path.exists():
    db_path.unlink()

shutil.copy(drive_db, db_path)

# Connect to db
conn = duckdb.connect(str(db_path))

print("\nDatabase loaded successfully.\n")

conn.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema == 'marts'
    ORDER BY table_name
""").df()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Database loaded successfully.



,table_name
0,county_season_summary
1,planting_base
2,planting_inputs
3,planting_weather
4,yield_dataset


### Load Modeling Data

> Only rows that are flagged `True` in `use_for_modeling` column; the **relevant features** include:
*   Weather-related features during the growing period: total rainfall, rain days, longest dry spell, & average temperatures.
*   Inputs per acre: fertilizer, manure, & pesticides.
*   Farm & crop info: crop, county, soil type, irrigation, certified seed, farm size, planted area, season type (long or short rains)

> The following info is **left out** from the model:

*   **IDs**: the model might memorize farm_ids and planting_ids.
*   **Season_year**: The model might fail to generalize to future years.











In [19]:
data = conn.sql("""
    SELECT planting_id, season_year, yield_kg_per_acre,
           crop, county, soil_type, season_type,
           irrigated, certified_seed, used_manure, used_pesticide,
           size_acres, area_planted_acres,
           season_rain_mm, rain_days, max_dry_spell_days, mean_tmax_c, mean_tmin_c,
           fertilizer_kg_per_acre, n_fertilizer_applications,
           manure_kg_per_acre, pesticide_litres_per_acre
    FROM marts.yield_dataset
    WHERE use_for_modelling
""").df()

# Cols flagged as True/False to be converted as 1.0/0.0; missing records to appear as NaN.
flag_cols = ['irrigated', 'certified_seed', 'used_manure', 'used_pesticide']
for col in flag_cols:
  data[col] = data[col].map({True: 1.0, False: 0.0})

# Define features (both categorical and numerical features) and target
cat_cols = ["crop", "county", "soil_type", "season_type"]
num_cols = ["size_acres", "area_planted_acres", "season_rain_mm", "rain_days", "max_dry_spell_days",
            "mean_tmax_c", "mean_tmin_c", "fertilizer_kg_per_acre", "n_fertilizer_applications",
            "manure_kg_per_acre", "pesticide_litres_per_acre"]
features = cat_cols + flag_cols + num_cols
target = "yield_kg_per_acre"

# Magnitude of the data that is ideal for modeling
print(f"{len(data):,} rows, {len(features)} features")
# Missing values
data[features + [target]].isna().sum().loc[lambda s: s > 0].to_frame("missing_values")

4,451 rows, 19 features


,missing_values
soil_type,190
irrigated,119
certified_seed,272
size_acres,32


### Train-test split by time

> Data from 2019 to '22 is used for training and model selection. Seasons '23 and '24 are held back as the test set and used once, at the end. A random split would mix seasons together, and the model could learn each season's weather from its neighbours in the data.

> Missing values are filled inside the model pipeline, using only training data. If we filled them before splitting, information from the test set would leak into training.



In [34]:
# Train test split by time
train = data[data['season_year'] <= 2022]
test = data[data['season_year'] >= 2023]

X_train, X_test = train[features], test[features]
y_train, y_test = train[target], test[target]

print("Features dimensions:", X_train.shape, X_test.shape)
print("Target dimensions:", y_train.shape, y_test.shape)

Features dimensions: (2996, 19) (1455, 19)
Target dimensions: (2996,) (1455,)


### Define the Models

> All four models share the same preprocessing steps: fill missing numbers with the training median, fill missing categories with "unknown", and one-hot encode the categories. Additionally, linear regression also gets the numerical cols scaled.

*    **Baseline - crop median:** Predicts the typical yield for a crop. Other models are expected to beat the baseline.

*   **Ridge regression:**  A simple linear model. It can't capture the ideal rainfall values, so we expect it to struggle.
*   **Random forest:**  Many decision trees averaged together. Handles curves and interactions.
*   **Gradient boosting:** Decision trees built one after another, each correcting the last. Expected to be the strongest model.


